In [34]:
"""
=============================================================================
MVP-2 — Chequeo Diagnóstico: Top 10 Genes Elastic Net vs study_part
=============================================================================
 
Objetivo: Discriminar si los top genes del Elastic Net RNA capturan señal
biológica de envejecimiento (PDL) o artefacto de batch (study_part).
 
Para cada gen del top 10:
  1. ρ(gen, pdl_norm)           → ¿Correlaciona con aging?
  2. ANOVA gen ~ study_part     → ¿Difiere entre protocolos?
  3. Correlación parcial        → ¿Gen correlaciona con study_part
     gen ~ study_part | PDL        DESPUÉS de controlar PDL?
 
Criterio de clasificación:
  - |ρ(PDL)| > 0.3  AND  p_anova(study_part) > 0.05   → Señal biológica
  - |ρ(PDL)| < 0.2  AND  p_anova(study_part) < 0.001  → Batch artifact
  - Mixto                                               → Ambiguo (acoplado)
 
Autor: JCB (asistido por Claude)
Fecha: 1 de abril de 2026
Proyecto: Inferencia del envejecimiento celular — MVP-2
=============================================================================
"""
 
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
import json
from datetime import datetime
 
warnings.filterwarnings('ignore')

In [35]:
# =============================================================================
# 0. CONFIGURACIÓN — AJUSTAR RUTAS
# =============================================================================
 
DATA_DIR = Path("/Users/JCB/Documentos/Proyecto Integrador/data/")
MANIFEST_DIR = DATA_DIR / "manifests"
RESULTS_DIR = Path("/Users/JCB/Documentos/Proyecto Integrador/results_2/")
BASELINES_DIR = RESULTS_DIR / "mvp2_baselines"
 
# Archivos de entrada
MANIFEST_FILE = "manifest_mvp2_pretrain_20260328_143235.csv"
RNA_MATRIX_FILE = "mvp2_rna_selected_20260328_143235.csv.gz"
 
# Top 10 genes del Elastic Net (del estado_proyecto_mvp2_30mar2026.md)
TOP_10_GENES = [
    "CES1",       # Carboxylesterase 1 — metabolismo lipídico
    "PREX1",      # Rac signaling — citoesqueleto
    "ZNF300P1",   # Pseudogene — ⚠️ sospechoso
    "PDE1C",      # Phosphodiesterase — cAMP
    "BCL11B",     # Transcription factor — desarrollo
    "PPP1R14C",   # Protein phosphatase — señalización
    "SFRP1",      # Wnt inhibitor — senescencia ✅
    "SPP1",       # Osteopontin — SASP-like ✅
    "TNFRSF21",   # TNF receptor — apoptosis
    "STMN2",      # Stathmin 2 — microtubules
]
 
# Los 40 hallmark genes forzados (para referencia de overlap)
HALLMARK_GENES = {
    "senescence": ["CDKN2A", "CDKN1A", "TP53", "RB1", "GLB1", "LMNB1", "HMGA1", "HMGA2"],
    "sasp": ["IL6", "CXCL8", "IL1A", "IL1B", "MMP1", "MMP3", "MMP9", "SERPINE1", "CCL2",
             "IGFBP3", "IGFBP5", "IGFBP7"],
    "mitochondrial": ["TFAM", "PPARGC1A", "SOD2", "PINK1"],
    "telomere": ["TERT", "TERC", "DKC1", "POT1"],
    "genomic_instability": ["ATM", "ATR", "BRCA1", "BRCA2", "CHEK1", "CHEK2"],
    "nutrient_sensing": ["MTOR", "SIRT1", "SIRT3", "SIRT6", "IGF1", "IGF1R"],
}
ALL_HALLMARK = [g for gs in HALLMARK_GENES.values() for g in gs]
 
print("=" * 70)
print("MVP-2 — Chequeo: Top 10 Genes EN vs study_part")
print("=" * 70)
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()

MVP-2 — Chequeo: Top 10 Genes EN vs study_part
Fecha: 2026-04-01 14:15



In [36]:
# =============================================================================
# 1. CARGAR DATOS
# =============================================================================
 
print("1. Cargando datos...")
 
# Manifest PRETRAIN (tiene study_part, pdl_norm, rna_matrix_col, fold)
manifest = pd.read_csv(MANIFEST_DIR / MANIFEST_FILE)
print(f"   Manifest: {manifest.shape}")
 
# Filtrar solo muestras con RNA
manifest_rna = manifest[manifest['has_rna'] == True].copy()
print(f"   Muestras con RNA: {manifest_rna.shape[0]}")
 
# Matriz RNA
rna_matrix = pd.read_csv(MANIFEST_DIR / RNA_MATRIX_FILE, compression='gzip', index_col=0)
print(f"   Matriz RNA: {rna_matrix.shape}")
 
# ⚠️ CORRECCIÓN: La matriz está en formato (muestras × genes)
# Necesitamos transponer para tener (genes × muestras)
rna_matrix = rna_matrix.T
print(f"   Matriz RNA (transpuesta): {rna_matrix.shape}")

# Verificar que top 10 genes están en la matriz
genes_found = [g for g in TOP_10_GENES if g in rna_matrix.index]
genes_missing = [g for g in TOP_10_GENES if g not in rna_matrix.index]
print(f"   Genes encontrados: {len(genes_found)}/{len(TOP_10_GENES)}")
if genes_missing:
    print(f"   ⚠️ Genes NO encontrados: {genes_missing}")

1. Cargando datos...
   Manifest: (715, 50)
   Muestras con RNA: 345
   Matriz RNA: (345, 2027)
   Matriz RNA (transpuesta): (2027, 345)
   Genes encontrados: 10/10


In [37]:
# =============================================================================
# DEBUG: Inspeccionar genes en la matriz
# =============================================================================

print("\n🔍 DEBUG: Primeros 50 genes en la matriz RNA:")
print(list(rna_matrix.index[:50]))

print(f"\n🔍 Total de genes en matriz: {len(rna_matrix.index)}")

# Buscar si los nombres están en diferentes formatos
print("\n🔍 Buscando variaciones de TOP_10_GENES:")
for gene in TOP_10_GENES:
    # Buscar exact match
    exact = gene in rna_matrix.index
    # Buscar case-insensitive
    lower = gene.lower() in [g.lower() for g in rna_matrix.index]
    # Buscar como substring
    substring_matches = [g for g in rna_matrix.index if gene.lower() in g.lower()]
    
    print(f"   {gene:12} - exact: {exact}, lower: {lower}, matches: {substring_matches[:3]}")



🔍 DEBUG: Primeros 50 genes en la matriz RNA:
['A2M', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA13', 'ABCA3', 'ABCA6', 'ABCA8', 'ABCB4', 'ABI3BP', 'ABLIM1', 'ABLIM2', 'ABLIM3', 'ABTB2', 'ACAN', 'ACAT2', 'ACE', 'ACKR1', 'ACKR3', 'ACKR4', 'ACOT7', 'ACP5', 'ACSS1', 'ACTA2', 'ACTA2-AS1', 'ACTC1', 'ACTG2', 'ACVR2A', 'ADA', 'ADAM12', 'ADAM19', 'ADAM23', 'ADAMTS12', 'ADAMTS14', 'ADAMTS15', 'ADAMTS3', 'ADAMTS5', 'ADAMTS6', 'ADAMTS7', 'ADAMTS9', 'ADAMTS9-AS2', 'ADAMTSL1', 'ADAMTSL4', 'ADARB1', 'ADCY1', 'ADCY4', 'ADGRB2', 'ADGRD1', 'ADGRG6', 'ADGRL2']

🔍 Total de genes en matriz: 2027

🔍 Buscando variaciones de TOP_10_GENES:
   CES1         - exact: True, lower: True, matches: ['CES1']
   PREX1        - exact: True, lower: True, matches: ['PREX1']
   ZNF300P1     - exact: True, lower: True, matches: ['ZNF300P1']
   PDE1C        - exact: True, lower: True, matches: ['PDE1C']
   BCL11B       - exact: True, lower: True, matches: ['BCL11B']
   PPP1R14C     - exact: True, lower: True, matches: ['PPP1R14C']
   

In [38]:
# =============================================================================
# 2. MERGE: expresión de genes + metadata
# =============================================================================
 
print("\n2. Construyendo tabla de análisis...")
 
# Transponer RNA: columnas = genes, filas = muestras (Sample_N)
rna_T = rna_matrix.loc[genes_found].T  # (345 muestras × N genes)
rna_T.index.name = 'rna_matrix_col'
rna_T = rna_T.reset_index()
 
# Merge con manifest usando rna_matrix_col
# NOTA: rna_matrix_col en manifest tiene formato "Sample_21", 
#       y las columnas de rna_matrix son también "Sample_21"
df = manifest_rna.merge(rna_T, on='rna_matrix_col', how='inner')
print(f"   Tabla merged: {df.shape[0]} muestras × {df.shape[1]} columnas")
 
# Verificar columnas necesarias
required_cols = ['pdl_norm', 'study_part', 'cell_line', 'fold']
for col in required_cols:
    assert col in df.columns, f"Columna '{col}' no encontrada en manifest"
    print(f"   ✓ {col}: {df[col].nunique()} valores únicos")
 
# Limpiar: excluir NaN en study_part o pdl_norm
n_before = len(df)
df = df.dropna(subset=['pdl_norm', 'study_part'])
print(f"   Muestras tras limpiar NaN: {len(df)} (eliminadas: {n_before - len(df)})")


2. Construyendo tabla de análisis...
   Tabla merged: 345 muestras × 60 columnas
   ✓ pdl_norm: 331 valores únicos
   ✓ study_part: 3 valores únicos
   ✓ cell_line: 7 valores únicos
   ✓ fold: 3 valores únicos
   Muestras tras limpiar NaN: 339 (eliminadas: 6)


In [39]:
# =============================================================================
# 3. ANÁLISIS POR GEN
# =============================================================================
 
print("\n" + "=" * 70)
print("3. RESULTADOS POR GEN")
print("=" * 70)
 
results = []
 
for gene in genes_found:
    print(f"\n--- {gene} ---")
    
    expr = df[gene].values
    pdl = df['pdl_norm'].values
    sp = df['study_part'].values
    
    # 3a. Correlación Spearman con PDL
    rho_pdl, p_pdl = stats.spearmanr(expr, pdl)
    
    # 3b. ANOVA: expresión ~ study_part
    groups_sp = [expr[sp == s] for s in np.unique(sp)]
    f_anova, p_anova = stats.f_oneway(*groups_sp)
    
    # 3c. Eta-squared (tamaño del efecto para ANOVA)
    grand_mean = np.mean(expr)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups_sp)
    ss_total = np.sum((expr - grand_mean)**2)
    eta_sq = ss_between / ss_total if ss_total > 0 else 0
    
    # 3d. Correlación parcial: gen ~ study_part | PDL
    # Método: residualizar ambas variables respecto a PDL, luego correlacionar
    # Usamos regresión lineal simple para obtener residuos
    
    # Para study_part categórico, usamos point-biserial / ANCOVA simplificado:
    # Residuos de gen ~ PDL
    slope_g, intercept_g, _, _, _ = stats.linregress(pdl, expr)
    residuals_gene = expr - (slope_g * pdl + intercept_g)
    
    # ANOVA sobre residuos: ¿study_part explica varianza residual?
    groups_resid = [residuals_gene[sp == s] for s in np.unique(sp)]
    f_resid, p_resid = stats.f_oneway(*groups_resid)
    
    # Eta-squared residual
    grand_mean_r = np.mean(residuals_gene)
    ss_between_r = sum(len(g) * (np.mean(g) - grand_mean_r)**2 for g in groups_resid)
    ss_total_r = np.sum((residuals_gene - grand_mean_r)**2)
    eta_sq_resid = ss_between_r / ss_total_r if ss_total_r > 0 else 0
    
    # 3e. Clasificación
    if abs(rho_pdl) > 0.3 and p_anova > 0.05:
        classification = "🟢 BIOLÓGICO"
    elif abs(rho_pdl) < 0.2 and p_anova < 0.001:
        classification = "🔴 BATCH ARTIFACT"
    elif abs(rho_pdl) > 0.3 and p_anova < 0.001:
        classification = "🟡 ACOPLADO (bio + batch)"
    elif p_resid > 0.05:
        classification = "🟢 BIO (batch desaparece tras controlar PDL)"
    else:
        classification = "🟡 AMBIGUO"
    
    # 3f. Medias por study_part (para ver dirección)
    sp_means = {str(int(s)): f"{np.mean(expr[sp == s]):.2f}" 
                for s in sorted(np.unique(sp))}
    
    print(f"  ρ(PDL):     {rho_pdl:+.3f}  (p={p_pdl:.1e})")
    print(f"  ANOVA sp:   F={f_anova:.1f}, p={p_anova:.1e}, η²={eta_sq:.3f}")
    print(f"  ANOVA|PDL:  F={f_resid:.1f}, p={p_resid:.1e}, η²={eta_sq_resid:.3f}")
    print(f"  Medias sp:  {sp_means}")
    print(f"  → {classification}")
    
    results.append({
        'gene': gene,
        'rho_pdl': round(rho_pdl, 4),
        'p_pdl': p_pdl,
        'f_anova_sp': round(f_anova, 2),
        'p_anova_sp': p_anova,
        'eta_sq_sp': round(eta_sq, 4),
        'f_anova_resid': round(f_resid, 2),
        'p_anova_resid': p_resid,
        'eta_sq_resid': round(eta_sq_resid, 4),
        'classification': classification,
        'sp_means': sp_means,
    })


3. RESULTADOS POR GEN

--- CES1 ---
  ρ(PDL):     -0.829  (p=6.5e-87)
  ANOVA sp:   F=47.5, p=6.9e-19, η²=0.220
  ANOVA|PDL:  F=9.6, p=9.2e-05, η²=0.054
  Medias sp:  {'2': '5.18', '3': '6.87', '4': '6.05'}
  → 🟡 ACOPLADO (bio + batch)

--- PREX1 ---
  ρ(PDL):     -0.438  (p=2.6e-17)
  ANOVA sp:   F=7.3, p=7.8e-04, η²=0.042
  ANOVA|PDL:  F=32.2, p=1.6e-13, η²=0.161
  Medias sp:  {'2': '5.94', '3': '5.66', '4': '6.15'}
  → 🟡 ACOPLADO (bio + batch)

--- ZNF300P1 ---
  ρ(PDL):     -0.598  (p=2.8e-34)
  ANOVA sp:   F=23.1, p=4.0e-10, η²=0.121
  ANOVA|PDL:  F=3.9, p=2.2e-02, η²=0.023
  Medias sp:  {'2': '5.23', '3': '5.86', '4': '5.61'}
  → 🟡 ACOPLADO (bio + batch)

--- PDE1C ---
  ρ(PDL):     +0.534  (p=2.3e-26)
  ANOVA sp:   F=17.1, p=8.2e-08, η²=0.093
  ANOVA|PDL:  F=48.2, p=4.1e-19, η²=0.223
  Medias sp:  {'2': '10.72', '3': '10.75', '4': '11.50'}
  → 🟡 ACOPLADO (bio + batch)

--- BCL11B ---
  ρ(PDL):     -0.563  (p=1.0e-29)
  ANOVA sp:   F=33.5, p=5.4e-14, η²=0.166
  ANOVA|PDL:  F=10.

In [40]:
# =============================================================================
# 4. TABLA RESUMEN
# =============================================================================
 
print("\n" + "=" * 70)
print("4. TABLA RESUMEN")
print("=" * 70)
 
df_results = pd.DataFrame(results)
summary_cols = ['gene', 'rho_pdl', 'p_anova_sp', 'eta_sq_sp', 
                'p_anova_resid', 'eta_sq_resid', 'classification']
print(df_results[summary_cols].to_string(index=False))
 
# Conteo de clasificaciones
print("\n--- Conteo de clasificaciones ---")
for clf in df_results['classification'].unique():
    n = (df_results['classification'] == clf).sum()
    genes_in = df_results[df_results['classification'] == clf]['gene'].tolist()
    print(f"  {clf}: {n} genes → {genes_in}")


4. TABLA RESUMEN
    gene  rho_pdl   p_anova_sp  eta_sq_sp  p_anova_resid  eta_sq_resid                              classification
    CES1  -0.8286 6.859912e-19     0.2204   9.155585e-05        0.0538                    🟡 ACOPLADO (bio + batch)
   PREX1  -0.4379 7.810471e-04     0.0417   1.551246e-13        0.1610                    🟡 ACOPLADO (bio + batch)
ZNF300P1  -0.5982 4.012093e-10     0.1208   2.174529e-02        0.0225                    🟡 ACOPLADO (bio + batch)
   PDE1C   0.5339 8.191593e-08     0.0926   4.085067e-19        0.2228                    🟡 ACOPLADO (bio + batch)
  BCL11B  -0.5628 5.429070e-14     0.1662   4.202235e-05        0.0582                    🟡 ACOPLADO (bio + batch)
PPP1R14C   0.6054 4.138882e-09     0.1085   1.721310e-01        0.0104                    🟡 ACOPLADO (bio + batch)
   SFRP1  -0.1143 6.122920e-01     0.0029   9.630537e-01        0.0002 🟢 BIO (batch desaparece tras controlar PDL)
    SPP1   0.4130 3.077909e-04     0.0470   5.216213e-02      

In [41]:
# =============================================================================
# 5. ANÁLISIS COMPLEMENTARIO: Estabilidad entre folds
# =============================================================================
 
print("\n" + "=" * 70)
print("5. CORRELACIÓN ρ(gen, PDL) POR FOLD")
print("=" * 70)
print("(Verifica si la señal es consistente o fold-dependiente)")
print()
 
folds = sorted(df['fold'].dropna().unique())
 
header = f"{'Gene':<12}" + "".join(f"{'Fold '+str(int(f)):>12}" for f in folds) + f"{'Global':>12}"
print(header)
print("-" * len(header))
 
for gene in genes_found:
    row = f"{gene:<12}"
    for fold in folds:
        mask = df['fold'] == fold
        if mask.sum() > 5:
            r, _ = stats.spearmanr(df.loc[mask, gene], df.loc[mask, 'pdl_norm'])
            row += f"{r:>+12.3f}"
        else:
            row += f"{'N/A':>12}"
    # Global
    r_global, _ = stats.spearmanr(df[gene], df['pdl_norm'])
    row += f"{r_global:>+12.3f}"
    print(row)


5. CORRELACIÓN ρ(gen, PDL) POR FOLD
(Verifica si la señal es consistente o fold-dependiente)

Gene              Fold 0      Fold 1      Fold 2      Global
------------------------------------------------------------
CES1              -0.916      -0.802      -0.632      -0.829
PREX1             -0.453      -0.509      -0.247      -0.438
ZNF300P1          -0.674      -0.683      -0.358      -0.598
PDE1C             +0.300      +0.711      +0.549      +0.534
BCL11B            -0.831      -0.527      -0.263      -0.563
PPP1R14C          +0.598      +0.598      +0.611      +0.605
SFRP1             -0.232      -0.171      -0.076      -0.114
SPP1              +0.329      +0.429      +0.458      +0.413
TNFRSF21          -0.378      -0.057      -0.434      -0.144
STMN2             -0.305      -0.254      -0.465      -0.296


In [42]:
# =============================================================================
# 6. ANÁLISIS COMPLEMENTARIO: Overlap con hallmarks
# =============================================================================
 
print("\n" + "=" * 70)
print("6. OVERLAP TOP 10 vs HALLMARK GENES")
print("=" * 70)
 
overlap = [g for g in TOP_10_GENES if g in ALL_HALLMARK]
print(f"  Overlap: {len(overlap)}/10 genes")
if overlap:
    for g in overlap:
        cat = [k for k, v in HALLMARK_GENES.items() if g in v][0]
        print(f"    {g} → categoría: {cat}")
else:
    print("  Ningún gen del top 10 es un hallmark gene predefinido.")
    print("  Genes aging-related parciales (del estado del proyecto):")
    partial_aging = ["SFRP1", "SPP1", "TGFB2", "TXNIP", "CCND2", "CDC25C"]
    for g in partial_aging:
        if g in TOP_10_GENES:
            print(f"    ✅ {g} está en top 10")
        else:
            print(f"    ⚪ {g} está en top 50 pero no top 10")



6. OVERLAP TOP 10 vs HALLMARK GENES
  Overlap: 0/10 genes
  Ningún gen del top 10 es un hallmark gene predefinido.
  Genes aging-related parciales (del estado del proyecto):
    ✅ SFRP1 está en top 10
    ✅ SPP1 está en top 10
    ⚪ TGFB2 está en top 50 pero no top 10
    ⚪ TXNIP está en top 50 pero no top 10
    ⚪ CCND2 está en top 50 pero no top 10
    ⚪ CDC25C está en top 50 pero no top 10


In [ ]:
# =============================================================================
# 7. GUARDAR RESULTADOS
# =============================================================================
 
print("\n" + "=" * 70)
print("7. GUARDANDO RESULTADOS")
print("=" * 70)
 
output_dir = BASELINES_DIR
output_dir.mkdir(parents=True, exist_ok=True)
 
# CSV con resultados
csv_path = output_dir / "diagnostico_genes_vs_studypart.csv"
df_results.to_csv(csv_path, index=False)
print(f"  ✓ CSV: {csv_path}")
 
# JSON con metadata
meta = {
    "analysis": "Top 10 EN genes vs study_part diagnostic",
    "date": datetime.now().isoformat(),
    "n_samples": len(df),
    "n_genes_analyzed": len(genes_found),
    "genes_missing": genes_missing,
    "classification_counts": df_results['classification'].value_counts().to_dict(),
    "conclusion": "PENDING — review classification counts",
}
json_path = output_dir / "diagnostico_genes_vs_studypart_meta.json"
with open(json_path, 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f"  ✓ JSON: {json_path}")


In [32]:
# =============================================================================
# 8. DIAGNÓSTICO FINAL
# =============================================================================
 
print("\n" + "=" * 70)
print("8. DIAGNÓSTICO FINAL")
print("=" * 70)
 
n_bio = sum('BIOLÓGICO' in c or 'BIO' in c for c in df_results['classification'])
n_batch = sum('BATCH' in c for c in df_results['classification'])
n_coupled = sum('ACOPLADO' in c for c in df_results['classification'])
n_ambig = sum('AMBIGUO' in c for c in df_results['classification'])
 
print(f"""
  Genes biológicos:     {n_bio}/10
  Genes batch artifact: {n_batch}/10
  Genes acoplados:      {n_coupled}/10
  Genes ambiguos:       {n_ambig}/10
""")
 
if n_batch >= 5:
    print("  ⛔ ALERTA: Mayoría de genes top son batch artifacts.")
    print("     → Baseline EN puede estar inflado por study_part.")
    print("     → Batch probe en Exp 2 es CRÍTICO.")
    print("     → Considerar incluir study_part como covariable en encoder.")
elif n_coupled >= 5:
    print("  ⚠️ PRECAUCIÓN: Mayoría de genes están acoplados (bio + batch).")
    print("     → Señal es real pero confundida con protocolo.")
    print("     → Batch probe condicional discriminará.")
    print("     → La narrativa correcta: 'señal biológica captada en")
    print("       un contexto experimental específico'.")
elif n_bio >= 5:
    print("  ✅ POSITIVO: Mayoría de genes son biológicos.")
    print("     → Baseline EN captura señal de aging, no batch.")
    print("     → Batch probe en Exp 2 sigue siendo necesario")
    print("       pero expectativa es ΔAUC bajo.")
else:
    print("  🟡 MIXTO: Resultados heterogéneos.")
    print("     → Analizar caso por caso.")
    print("     → ZNF300P1 (pseudogene) merece atención especial.")
 
print("\n" + "=" * 70)
print("FIN DEL CHEQUEO")
print("=" * 70)
 


8. DIAGNÓSTICO FINAL

  Genes biológicos:     2/10
  Genes batch artifact: 1/10
  Genes acoplados:      7/10
  Genes ambiguos:       0/10

  ⚠️ PRECAUCIÓN: Mayoría de genes están acoplados (bio + batch).
     → Señal es real pero confundida con protocolo.
     → Batch probe condicional discriminará.
     → La narrativa correcta: 'señal biológica captada en
       un contexto experimental específico'.

FIN DEL CHEQUEO
